In [1]:
import numpy as np
import pyslha
import glob
from collections import namedtuple


### Get Widths

In [11]:
pars = namedtuple('parameters',['M1','M0','lambdaUV','w1'])
parameters = []
w_all = set([])
for f in glob.glob('./width_results3/*.dat'):
    slhaData = pyslha.readSLHAFile(f)
    p = pars(M1=slhaData.blocks['MASS'][4000023], M0=slhaData.blocks['MASS'][4000022], lambdaUV=slhaData.blocks['NPINPUTS'][1], w1=slhaData.decays[4000023].totalwidth)
    w_all.add(p.w1)
    parameters.append(p)

In [12]:
mSs = [400, 450, 600, 800, 1000] #mass of scalar particle [GeV]
# beta1s = [0.1, 0.2, 0.25, 0.3, 0.4] #boost of the chi1 in the labframe
beta1s = [0.5,0.6,0.8] #boost of the chi1 in the labframe
Dm21s = [5] #mass difference between chi2 and chi1 [GeV]
Dm10s = [60, 100] #mass difference between chi1 and chi0 [GeV] (come back and try 100)
ctau0s = [20] #proper lifetime of Chi1 [m]

In [14]:
pts = []
lambdaUV0 = 9.07e6
for mS in mSs:
    for beta1 in beta1s:
        for Dm21 in Dm21s:
            m1_solutions = [ (-beta1 ** 2 * Dm21 ** 3 + sgn * (-beta1 ** 2 * Dm21 ** 4 * mS ** 2 + 2 * beta1 ** 2 * Dm21 ** 2 * mS ** 4 - beta1 ** 2 * mS ** 6 + Dm21 ** 4 * mS ** 2 - 2 * Dm21 ** 2 * mS ** 4 + mS ** 6) ** .5 + beta1 ** 2 * Dm21 * mS ** 2 + Dm21 ** 3 - Dm21 * mS ** 2)/(2 * (beta1 ** 2 * Dm21 ** 2 - Dm21 ** 2 + mS ** 2)) for sgn in [+1 , -1]]
            M1 = max(m1_solutions)
            M2 = M1 + Dm21 #mass of chi2 [GeV]
            for Dm10 in Dm10s:
                M0 = M1 - Dm10 #mass of chi0 [GeV]
                if sorted([mS,M2,M1,M0],reverse=True) != [mS,M2,M1,M0]:
                    print(f"Skipping point: mS={mS}, M2={M2}, M1={M1}, M0={M0} because it does not satisfy mS > M2 > M1 > M0")
                    continue
                if M1-M0 < 15.0:
                    continue
                p = [par for par in parameters if np.isclose(par.M1, M1,rtol=1e-3) and np.isclose(par.M0, M0,rtol=1e-3)]
                if not p:
                    print(f"Skipping point: mS={mS}, M2={M2}, M1={M1}, M0={M0} because it does not have a corresponding parameter file")
                    continue
                p = p[0]
                w10 = p.w1
                lambdaUV0 = p.lambdaUV
                for ctau0_m in ctau0s:
                    w1 = 6.58*3.e-17/ctau0_m
                    lambdaUV = lambdaUV0/np.sqrt(w1/w10)
                    # lambdaUV = lambdaUV0
                    pt = [float(f"{x:1.4g}") if x < 1e4 else float(f"{x:1.4e}") for x in (mS, M2, M1, M0, lambdaUV, ctau0_m)]
                    pts.append(tuple(pt))
                    

In [15]:
print(len(pts))
print(pts)

30
[(400.0, 176.3, 171.3, 111.3, 8681700.0, 20.0), (400.0, 176.3, 171.3, 71.32, 35816000.0, 20.0), (400.0, 163.4, 158.4, 98.39, 8408800.0, 20.0), (400.0, 163.4, 158.4, 58.39, 33258000.0, 20.0), (400.0, 124.1, 119.1, 59.09, 7254000.0, 20.0), (400.0, 124.1, 119.1, 19.09, 21071000.0, 20.0), (450.0, 198.0, 193.0, 133.0, 9028600.0, 20.0), (450.0, 198.0, 193.0, 92.97, 39376000.0, 20.0), (450.0, 183.4, 178.4, 118.4, 8780000.0, 20.0), (450.0, 183.4, 178.4, 78.39, 37108000.0, 20.0), (450.0, 139.1, 134.1, 74.09, 7792600.0, 20.0), (450.0, 139.1, 134.1, 34.09, 26683000.0, 20.0), (600.0, 262.9, 257.9, 197.9, 9745000.0, 20.0), (600.0, 262.9, 257.9, 157.9, 46131000.0, 20.0), (600.0, 243.4, 238.4, 178.4, 9551300.0, 20.0), (600.0, 243.4, 238.4, 138.4, 44524000.0, 20.0), (600.0, 184.1, 179.1, 119.1, 8785000.0, 20.0), (600.0, 184.1, 179.1, 79.09, 37224000.0, 20.0), (800.0, 349.5, 344.5, 284.5, 10244000.0, 20.0), (800.0, 349.5, 344.5, 244.5, 51185000.0, 20.0), (800.0, 323.4, 318.4, 258.4, 10137000.0, 20.0

In [9]:
pts_m1 = []
for pt in pts:
    p = (pt[2],pt[3], pt[4])
    if p not in pts_m1:
        pts_m1.append(p)

In [10]:
print(len(pts_m1))
print(pts_m1)

30
[(171.3, 111.3, 9070000.0), (171.3, 71.32, 9070000.0), (158.4, 98.39, 9070000.0), (158.4, 58.39, 9070000.0), (119.1, 59.09, 9070000.0), (119.1, 19.09, 9070000.0), (193.0, 133.0, 9070000.0), (193.0, 92.97, 9070000.0), (178.4, 118.4, 9070000.0), (178.4, 78.39, 9070000.0), (134.1, 74.09, 9070000.0), (134.1, 34.09, 9070000.0), (257.9, 197.9, 9070000.0), (257.9, 157.9, 9070000.0), (238.4, 178.4, 9070000.0), (238.4, 138.4, 9070000.0), (179.1, 119.1, 9070000.0), (179.1, 79.09, 9070000.0), (344.5, 284.5, 9070000.0), (344.5, 244.5, 9070000.0), (318.4, 258.4, 9070000.0), (318.4, 218.4, 9070000.0), (239.1, 179.1, 9070000.0), (239.1, 139.1, 9070000.0), (431.1, 371.1, 9070000.0), (431.1, 331.1, 9070000.0), (398.4, 338.4, 9070000.0), (398.4, 298.4, 9070000.0), (299.1, 239.1, 9070000.0), (299.1, 199.1, 9070000.0)]
